# Agentic RAG using `create_agent` — Simple Version

This notebook recreates the **complete Agentic RAG flow** from `agentic_rag(4).ipynb`, but replaces the custom `StateGraph` + `ToolNode` + manual agent loop with LangChain's current `create_agent`.

The flow is intentionally simple:

**Route → Decompose → Create Agent Retrieval → Retrieval Limit → Collect Docs → Evaluate → Rewrite → Build Prompt → Generate**

`create_agent` handles the internal:

**LLM → Tool → LLM → Tool → ... → Final response**

loop automatically.


In [1]:
import os
from pydantic import BaseModel

from dotenv import load_dotenv
from ddgs import DDGS
from trafilatura import fetch_url, extract

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma


C:\Users\Uttam\AppData\Local\Temp\ipykernel_11340\153396262.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is missing")


## 1. Load and split the internal PDF

In [3]:
loader = PyPDFLoader("../documents/evs_oil_price_shock.pdf")

raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} pages")


incorrect startxref pointer(1)
parsing for Object Streams


Loaded 15 pages


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
)

chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")


Split into 51 chunks


## 2. Create the vector store

In [5]:
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=HF_TOKEN,
)

vectorstore = Chroma(
    collection_name="rag_create_agent",
    embedding_function=embeddings,
)

vectorstore.add_documents(chunks)

print("Vector store ready")


Vector store ready


## 3. Create the LLMs

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
)

agent_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
)


## 4. Retrieval tools

There are still exactly two retrieval capabilities:

- `vector_store_search` → internal EV/oil report
- `web_search` → current web information


In [7]:
@tool(response_format="content_and_artifact")
def vector_store_search(query: str, k: int = 3):
    "Search the internal EV/oil report."
    docs = vectorstore.similarity_search(query, k=k)

    context = (
        "## Vector Store Results\n\n"
        + "\n\n".join(doc.page_content for doc in docs)
    )

    return context, docs


In [8]:
@tool(response_format="content_and_artifact")
def web_search(query: str, max_results: int = 3):
    "Search the web for current or real-time information."
    search_results = DDGS().text(query, max_results=max_results)

    docs = []

    for result in search_results or []:
        url = result.get("href", "")
        title = result.get("title", "")

        if not url:
            continue

        try:
            downloaded = fetch_url(url)
            content = extract(downloaded, output_format="markdown")

            if content:
                docs.append(
                    Document(
                        page_content=content,
                        metadata={
                            "source": url,
                            "title": title,
                        },
                    )
                )
        except Exception:
            continue

    context = (
        "## Web Search Results\n\n"
        + "\n\n".join(doc.page_content for doc in docs)
    )

    return context, docs


## 5. Create the agent with `create_agent`

This replaces the old:

`agent()` + `bind_tools()` + `ToolNode` + manual routing

with one simple agent.


In [9]:
AGENT_SYSTEM_PROMPT = (
    "You are a retrieval agent with access to two tools.\n\n"
    "1. vector_store_search — use this for questions that can be answered from the internal document: "
    "a technical report titled 'Will EVs Dampen the Oil Price Shock?' covering EV adoption trajectories, "
    "oil demand displacement scenarios, fleet turnover dynamics, battery cost trends, OPEC+ supply behavior, "
    "and energy price volatility projections through 2050. "
    "Use this tool whenever the query references the report, its findings, its projections, or any topic "
    "that would plausibly appear in this domain-specific report.\n\n"
    "2. web_search — use this for current or real-time information not covered by the document, "
    "such as recent market data, news, or statistics. "
    "Rephrase the need into a concise search query before using this tool.\n\n"
    "You may call one tool, both tools, or no tool. "
    "When both document knowledge and current data are relevant, call both tools.\n\n"
    "If the query contains numbered steps, treat every step as a separate retrieval task. "
    "Do not skip any step."
)

agent = create_agent(
    model=agent_llm,
    tools=[vector_store_search, web_search],
    system_prompt=AGENT_SYSTEM_PROMPT,
)


## 6. Route the question

Same logic as the original notebook:

- `True` → retrieval is needed
- `False` → go directly to generation


In [10]:
class RouteDecision(BaseModel):
    needs_retrieval: bool


def route_question(question: str) -> bool:
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Decide whether the question needs retrieval from a specialized document or the web. "
            "Return True when retrieval is needed; otherwise return False.",
        ),
        ("human", "{question}"),
    ])

    chain = prompt | llm.with_structured_output(RouteDecision)

    result = chain.invoke({"question": question})

    return result.needs_retrieval


## 7. Check for query decomposition

In [11]:
class DecompositionDecision(BaseModel):
    needs_decomposition: bool
    sub_queries: list[str]


def check_decomposition(question: str) -> DecompositionDecision:
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Determine whether the question asks for multiple distinct pieces of information "
            "that each require separate retrieval. "
            "If yes, create one focused sub-query for each piece. "
            "If no, return an empty sub_queries list.",
        ),
        ("human", "{question}"),
    ])

    chain = prompt | llm.with_structured_output(DecompositionDecision)

    return chain.invoke({"question": question})


## 8. Decompose the query when needed

In [12]:
def decompose_query(decision: DecompositionDecision) -> str:
    if not decision.needs_decomposition:
        return decision.sub_queries[0] if decision.sub_queries else ""

    steps = "\n".join(
        f"Step {i}: {query}"
        for i, query in enumerate(decision.sub_queries, start=1)
    )

    return (
        f"You must retrieve information for every step below before answering. "
        f"Do not skip any step.\n\n{steps}"
    )


## 9. Collect documents from `create_agent`

`create_agent` returns the full message history. Because the tools use `content_and_artifact`, the retrieved `Document` objects are available inside each `ToolMessage`.


In [13]:
def collect_docs(agent_result) -> list[Document]:
    docs = []

    for message in agent_result["messages"]:
        if isinstance(message, ToolMessage):
            artifact = getattr(message, "artifact", None)

            if artifact:
                docs.extend(artifact)

    return docs


def count_retrieval_steps(agent_result) -> int:
    # One AI message containing tool calls = one retrieval round.
    return sum(
        1
        for message in agent_result["messages"]
        if getattr(message, "tool_calls", None)
    )


## 10. Evaluate the retrieved documents

In [14]:
class RelevanceEvaluation(BaseModel):
    is_relevant: bool


def evaluate_docs(question: str, docs: list[Document]):
    if not docs:
        return False, []

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Decide whether the document passage directly helps answer the query. "
            "Return True only when it contains useful, specific facts or analysis.",
        ),
        ("human", "Query: {question}\n\nDocument:\n{doc}"),
    ])

    chain = prompt | llm.with_structured_output(RelevanceEvaluation)

    relevant_docs = []

    for doc in docs:
        result = chain.invoke({
            "question": question,
            "doc": doc.page_content,
        })

        if result.is_relevant:
            relevant_docs.append(doc)

    return bool(relevant_docs), relevant_docs


## 11. Rewrite the query when retrieval is not relevant

The original notebook allowed up to **3 rewrites**.


In [15]:
def rewrite_query(question: str) -> str:
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Rewrite the retrieval query to improve search quality. "
            "Use more specific terminology, different wording, or a narrower focus. "
            "Return only the rewritten query.",
        ),
        ("human", "Original query: {question}"),
    ])

    response = (prompt | llm).invoke({"question": question})

    return response.text.strip()


## 12. Retrieval-limit helper

The original graph counted retrieval rounds and stopped when the limit was reached.

With `create_agent`, the internal tool loop is controlled using `recursion_limit`. We pass only the **remaining number of retrieval rounds** each time.


In [16]:
def remaining_retrieval_steps(used: int, maximum: int) -> int:
    return max(maximum - used, 0)


## 13. Build the final prompt

In [17]:
def build_prompt(
    original_question: str,
    relevant_docs: list[Document],
    needs_decomposition: bool,
    rewrite_count: int,
):
    if rewrite_count >= 3 and not relevant_docs:
        return "FALLBACK"

    sections = ["You are a knowledgeable assistant."]

    if needs_decomposition:
        sections.append(
            "The question contained multiple parts. "
            "Answer every part in one coherent response."
        )

    context = "\n\n".join(doc.page_content for doc in relevant_docs)

    if context:
        web_docs = [
            doc for doc in relevant_docs
            if doc.metadata.get("source", "").startswith("http")
        ]

        pdf_docs = [
            doc for doc in relevant_docs
            if not doc.metadata.get("source", "").startswith("http")
        ]

        if web_docs and pdf_docs:
            sections.append(
                "Answer using the following context from the internal document and web search:"
            )
        elif web_docs:
            sections.append("Answer using the following context from web search:")
        else:
            sections.append("Answer using the following context from the internal document:")

        sections.append(f"Context:\n{context}")

        if web_docs:
            urls = "\n".join(
                f"- {doc.metadata['source']}"
                for doc in web_docs
                if doc.metadata.get("source")
            )
            if urls:
                sections.append(f"Sources:\n{urls}")

    else:
        sections.append(
            "No relevant documents were found. "
            "Answer from your general knowledge if possible."
        )

    return "\n\n".join(sections)


## 14. Generate the final answer

In [18]:
def generate_answer(question: str, prompt: str) -> str:
    if prompt == "FALLBACK":
        return (
            "I was unable to find relevant information to answer your query after multiple "
            "retrieval attempts. The knowledge source does not appear to contain content "
            "that addresses this question."
        )

    final_prompt = ChatPromptTemplate.from_messages([
        ("system", "{prompt}"),
        ("human", "{question}"),
    ])

    response = (final_prompt | llm).invoke({
        "prompt": prompt,
        "question": question,
    })

    return response.text


## 15. Complete Agentic RAG

This is the main function.

The custom graph from the original notebook becomes ordinary Python control flow:

**if + while + create_agent()**

The Agent itself handles the tool-calling loop.


In [19]:
def agentic_rag(question: str, max_retrieval_steps: int = 3):
    # 1. Route
    if not route_question(question):
        prompt = build_prompt(
            original_question=question,
            relevant_docs=[],
            needs_decomposition=False,
            rewrite_count=0,
        )

        return {
            "generation": generate_answer(question, prompt),
            "retrieval_count": 0,
            "rewrite_count": 0,
            "needs_decomposition": False,
            "is_relevant": False,
            "relevant_docs": [],
            "constructed_prompt": prompt,
        }

    # 2. Decompose
    decision = check_decomposition(question)
    needs_decomposition = decision.needs_decomposition

    retrieval_query = decompose_query(decision)

    if not retrieval_query:
        retrieval_query = question

    # 3. Retrieve -> Evaluate -> Rewrite
    retrieval_count = 0
    rewrite_count = 0
    relevant_docs = []

    while True:
        remaining = remaining_retrieval_steps(
            retrieval_count,
            max_retrieval_steps,
        )

        if remaining == 0:
            break

        result = agent.invoke(
            {"messages": [HumanMessage(content=retrieval_query)]},
            config={"recursion_limit": (remaining * 2) + 1},
        )

        retrieval_count += count_retrieval_steps(result)

        docs = collect_docs(result)

        is_relevant, relevant_docs = evaluate_docs(
            retrieval_query,
            docs,
        )

        if is_relevant:
            break

        if rewrite_count >= 3:
            break

        retrieval_query = rewrite_query(retrieval_query)
        rewrite_count += 1

    # 4. Build prompt
    prompt = build_prompt(
        original_question=question,
        relevant_docs=relevant_docs,
        needs_decomposition=needs_decomposition,
        rewrite_count=rewrite_count,
    )

    # 5. Generate
    answer = generate_answer(question, prompt)

    return {
        "generation": answer,
        "retrieval_count": retrieval_count,
        "rewrite_count": rewrite_count,
        "needs_decomposition": needs_decomposition,
        "is_relevant": bool(relevant_docs),
        "relevant_docs": relevant_docs,
        "constructed_prompt": prompt,
    }


## 16. Test 1 — Simple report question

In [20]:
query_1 = (
    "According to the report, by how many million barrels per day could a 300-million EV fleet "
    "displace oil demand by 2030?"
)

result_1 = agentic_rag(
    query_1,
    max_retrieval_steps=3,
)

print("=== Case 1 ===")
print("Needs decomposition:", result_1["needs_decomposition"])
print("Retrieval count:", result_1["retrieval_count"])
print("Rewrite count:", result_1["rewrite_count"])
print("Is relevant:", result_1["is_relevant"])
print("Relevant docs:", len(result_1["relevant_docs"]))
print("\nAnswer:\n", result_1["generation"])


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


=== Case 1 ===
Needs decomposition: False
Retrieval count: 1
Rewrite count: 0
Is relevant: True
Relevant docs: 1

Answer:
 According to the report, a fleet of 300 million EVs would displace approximately 4.6 million barrels of oil per day by 2030.


## 17. Test 2 — Complex multi-part question

In [22]:
query_2 = (
    "According to the report, how much have battery pack costs fallen over the past decade, "
    "and What about IPL 2027 and Who is Sponsoring IPL 2027 and What about -23dBm optical power in Syrotech ONT"
)

result_2 = agentic_rag(
    query_2,
    max_retrieval_steps=6,
)

print("=== Case 2 ===")
print("Needs decomposition:", result_2["needs_decomposition"])
print("Retrieval count:", result_2["retrieval_count"])
print("Rewrite count:", result_2["rewrite_count"])
print("Is relevant:", result_2["is_relevant"])
print("Relevant docs:", len(result_2["relevant_docs"]))
print("\nAnswer:\n", result_2["generation"])


=== Case 2 ===
Needs decomposition: True
Retrieval count: 1
Rewrite count: 0
Is relevant: True
Relevant docs: 6

Answer:
 Based on the provided documents, here are the answers to your questions:

### **Battery Pack Costs**
According to the research report, battery pack costs have fallen by approximately **90 percent** over the past decade. Specifically, costs dropped from approximately $1,200 per kilowatt-hour in 2010 to below $100 per kilowatt-hour in 2024.

### **IPL 2027 Details**
*   **Schedule:** The IPL 2027 (also known as IPL 19 or TATA IPL T20 2027) is expected to be held from **14 March 2027 to 25 May 2027** (though some sections of the text note a tentative window of 10 March to 15 May). The tournament will feature 10 teams playing 74 matches in a double round-robin and playoff format.
*   **Sponsorship:** The tournament is referred to as the **TATA IPL 2027**, indicating that **TATA** is the title sponsor.

### **-23dBm Optical Power in Syrotech ONT**
The provided documents 

## Original graph → `create_agent`

| Original `StateGraph` | Simplified `create_agent` version |
|---|---|
| `route_question` node | `route_question()` |
| `check_decomposition` node | `check_decomposition()` |
| `decompose_query` node | `decompose_query()` |
| `agent` node | `create_agent()` |
| `ToolNode` | handled internally by `create_agent()` |
| `check_retrieval_limit` | `recursion_limit` + remaining-step calculation |
| `collect_tool_output` | `collect_docs()` |
| `evaluate_docs` | `evaluate_docs()` |
| `rewrite_query` | `rewrite_query()` |
| `build_prompt` | `build_prompt()` |
| `generate` | `generate_answer()` |
| Conditional graph edges | normal Python `if` / `while` |

### Important

The **Agentic RAG logic is still complete**. What has been removed is only the custom graph plumbing. `create_agent` performs the model/tool loop internally.
